<a href="https://colab.research.google.com/github/Takayoshi-code/nanoGPT-class/blob/master/nanoGPT_Aozora_tiny.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## STEP1 nanoGPT環境をGitHubからダウンロードしてGoogle Corab上に展開します。

In [47]:
# ===== 安全版 =====

import os

print("===== Step 0: /content に移動 =====")
%cd /content

print("===== Step 1: クリーン =====")
!rm -rf nanoGPT
!rm -f master.zip

print("===== Step 2: ライブラリ =====")
!pip install -q sentencepiece datasets tqdm

print("===== Step 3: nanoGPT取得 =====")

!wget -q https://github.com/Takayoshi-code/nanoGPT-class/archive/refs/heads/master.zip
!unzip -q master.zip
!mv nanoGPT-class-master nanoGPT

print("===== Step 4: 移動 =====")
%cd /content/nanoGPT

print("===== Step 5: 確認 =====")
!ls

===== Step 0: /content に移動 =====
/content
===== Step 1: クリーン =====
===== Step 2: ライブラリ =====
===== Step 3: nanoGPT取得 =====
===== Step 4: 移動 =====
/content/nanoGPT
===== Step 5: 確認 =====
assets		 LICENSE		    scaling_laws.ipynb
bench.py	 model.py		    train.py
config		 nanoGPT_Aozora_tiny.ipynb  transformer_sizing.ipynb
configurator.py  README.md
data		 sample.py


##青空文庫収録作家一覧

In [20]:
import io
import zipfile
import requests
import pandas as pd

URL = "https://www.aozora.gr.jp/index_pages/list_person_all_extended_utf8.zip"

print("青空文庫のデータを取得中...")

r = requests.get(URL, timeout=60)
r.raise_for_status()

with zipfile.ZipFile(io.BytesIO(r.content)) as z:
    csv_name = [x for x in z.namelist() if x.endswith(".csv")][0]
    with z.open(csv_name) as f:
        df = pd.read_csv(f, low_memory=False)

df["作家名"] = (
    df["姓"].fillna("").astype(str).str.strip()
    + " "
    + df["名"].fillna("").astype(str).str.strip()
).str.strip()

authors = (
    df.groupby(["人物ID", "作家名"], as_index=False)
      .agg(公開作品数=("作品ID", "nunique"))
      .sort_values(["作家名", "人物ID"])
      .reset_index(drop=True)
)

print(f"\n収録作家数: {len(authors):,}人")
print("=" * 70)

for i, row in authors.iterrows():
    print(
        f"{i:4d}: {row['作家名']:<20} "
        f"人物ID={row['人物ID']:<6} "
        f"作品数={row['公開作品数']}"
    )

青空文庫のデータを取得中...

収録作家数: 1,335人
   0: Morishous T.H.E creative 人物ID=2255   作品数=2
   1: The Creative CAT     人物ID=1834   作品数=31
   2: hiro                 人物ID=1280   作品数=2
   3: sogo                 人物ID=913    作品数=3
   4: かぐつち みどり             人物ID=968    作品数=1
   5: ささき ふさ               人物ID=911    作品数=1
   6: とだ けん                人物ID=970    作品数=1
   7: アイプ・ロシディ             人物ID=2313   作品数=1
   8: アインシュタイン アルベルト       人物ID=1428   作品数=1
   9: アウンチェイン              人物ID=2396   作品数=1
  10: アポリネール ギヨーム          人物ID=2135   作品数=1
  11: アミーチス エドモンド・デ        人物ID=1048   作品数=1
  12: アラルコン ペドロ・アントニオ      人物ID=1716   作品数=1
  13: アリ サバハッティン           人物ID=2315   作品数=1
  14: アルチバシェッフ ミハイル・ペトローヴィチ 人物ID=366    作品数=3
  15: アルテンベルク ペーター         人物ID=1190   作品数=1
  16: アレニウス スヴァンテ          人物ID=226    作品数=1
  17: アレン ジェームズ            人物ID=1842   作品数=1
  18: アレン リリー・Ｌ            人物ID=1843   作品数=1
  19: アンデルセン ハンス・クリスチャン    人物ID=19     作品数=51
  20: アンドレーエフ レオニード・ニコラーエヴィチ 人物ID=907    作品数=2
  21: アークム フレデリ

## STEP2 作家、作品を選んでクリーニングして入力コーパスを作成します。

In [49]:
import io
import os
import re
import zipfile
import requests
import pandas as pd
from tqdm import tqdm
from bs4 import BeautifulSoup
from urllib.parse import urljoin

CSV_ZIP_URL = "https://www.aozora.gr.jp/index_pages/list_person_all_extended_utf8.zip"
BASE_DIR = "/content/nanoGPT/novel_selected"
RAW_OUTPUT = os.path.join(BASE_DIR, "selected_works.txt")
CLEAN_OUTPUT = os.path.join(BASE_DIR, "selected_works_clean.txt")
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; AozoraDownloader/1.0)"}

def normalize_name(s):
    if pd.isna(s):
        return ""
    return str(s).replace(" ", "").replace("　", "").strip()

def decode_aozora_text(raw):
    for enc in ["shift_jis", "cp932", "utf-8-sig", "utf-8"]:
        try:
            return raw.decode(enc)
        except UnicodeDecodeError:
            pass
    raise RuntimeError("文字コードを判定できませんでした。")

def clean_text(text):
    lines = text.split("\n")
    result = []
    started = False

    for line in lines:
        raw = line
        line = line.strip()

        if line.startswith("<|title|>"):
            continue
        if line.startswith("<|author|>"):
            continue

        if not started:
            if raw.startswith("　") and len(line) > 10 and "。" in line:
                started = True
            else:
                continue

        if (
            line.startswith("底本")
            or line.startswith("初出")
            or line.startswith("入力")
            or line.startswith("校正")
            or line.startswith("青空文庫")
            or line.startswith("作成ファイル")
            or "青空文庫作成ファイル" in line
        ):
            break

        if (
            "記号について" in line
            or "JIS" in line
            or "入力に使用" in line
            or "校正に使用" in line
        ):
            continue

        if re.fullmatch(r"[一二三四五六七八九十百千万]+", line):
            continue

        if re.fullmatch(r"[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩ]+", line):
            continue

        if line in {"上", "中", "下", "前編", "後編", "序", "跋", "附録", "解説"}:
            continue

        line = re.sub(r"｜", "", line)
        line = re.sub(r"《.*?》", "", line)
        line = re.sub(r"［＃.*?］", "", line)
        line = re.sub(r"〔.*?〕", "", line)
        line = re.sub(r"（注.*?）", "", line)
        line = re.sub(r"（[\d０-９]+）", "", line)

        line = line.replace("／＼", "")
        line = line.replace("※", "")
        line = line.replace("＊", "")

        if re.fullmatch(r"[0-9０-９A-Za-z]+", line):
            continue

        if len(line) < 2:
            continue

        result.append(line)

    text = "\n".join(result)
    text = re.sub(r"\n{3,}", "\n\n", text)

    if text.count("。") < 10:
        return ""

    if len(text) < 1000:
        return ""

    return text.strip()

def download_work(card_url):
    r = requests.get(card_url, headers=HEADERS, timeout=60)
    r.raise_for_status()
    r.encoding = r.apparent_encoding
    soup = BeautifulSoup(r.text, "html.parser")

    zip_candidates = []

    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.lower().split("?")[0].endswith(".zip"):
            url = urljoin(card_url, href)
            label = a.get_text(" ", strip=True)
            priority = 0 if "テキスト" in label else 1
            zip_candidates.append((priority, url))

    if not zip_candidates:
        return None

    zip_candidates.sort(key=lambda x: x[0])

    for _, zip_url in zip_candidates:
        try:
            r = requests.get(zip_url, headers=HEADERS, timeout=60)
            r.raise_for_status()

            with zipfile.ZipFile(io.BytesIO(r.content)) as z:
                txt_files = [name for name in z.namelist() if name.lower().endswith(".txt")]

                if not txt_files:
                    continue

                raw = z.read(txt_files[0])
                return decode_aozora_text(raw)

        except Exception:
            continue

    return None

def main():
    os.makedirs(BASE_DIR, exist_ok=True)

    print("青空文庫の作品一覧を取得しています...")

    r = requests.get(CSV_ZIP_URL, headers=HEADERS, timeout=60)
    r.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        csv_files = [name for name in z.namelist() if name.lower().endswith(".csv")]
        if not csv_files:
            raise RuntimeError("CSVが見つかりません。")
        with z.open(csv_files[0]) as f:
            df = pd.read_csv(f, low_memory=False)

    print(f"登録データ数: {len(df):,}")

    df["作家名"] = (
        df["姓"].fillna("").astype(str).str.strip()
        + " "
        + df["名"].fillna("").astype(str).str.strip()
    ).str.strip()

    df["作家名検索用"] = df["作家名"].apply(normalize_name)

    keyword = input("作家名を入力してください: ").strip()
    keyword_normalized = normalize_name(keyword)

    mask = df["作家名検索用"].str.contains(
        keyword_normalized,
        na=False,
        regex=False
    )

    authors = (
        df.loc[mask, ["人物ID", "作家名"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    if len(authors) == 0:
        raise ValueError(f"「{keyword}」に該当する作家が見つかりません。")

    print("\n=== 作家候補 ===")

    for i, row in authors.iterrows():
        print(f"{i:3d}: {row['作家名']} (人物ID={row['人物ID']})")

    if len(authors) == 1:
        n = 0
        print("候補が1人なので自動選択します。")
    else:
        while True:
            try:
                n = int(input("左端の選択番号を入力してください: "))
                if 0 <= n < len(authors):
                    break
            except ValueError:
                pass
            print(f"0～{len(authors)-1}を入力してください。")

    author_id = authors.iloc[n]["人物ID"]
    author_name = authors.iloc[n]["作家名"]

    print(f"\n選択した作家: {author_name} (人物ID={author_id})")

    columns = ["作品ID", "作品名", "図書カードURL"]

    if "副題" in df.columns:
        columns.insert(2, "副題")

    works = (
        df.loc[df["人物ID"] == author_id, columns]
        .drop_duplicates(subset=["作品ID"])
        .reset_index(drop=True)
    )

    print(f"\n=== {author_name} の作品一覧（{len(works)}作品）===")

    for i, row in works.iterrows():
        title = str(row["作品名"])

        if "副題" in works.columns and pd.notna(row["副題"]):
            subtitle = str(row["副題"]).strip()
            if subtitle:
                title += " ― " + subtitle

        print(f"{i:3d}: {title} (作品ID={row['作品ID']})")

    print("\n複数選択できます。")
    print("例: 0,3,5,8")
    print("全作品: all")

    selection = input("作品番号を入力してください: ").strip()

    if selection.lower() == "all":
        selected_indices = list(range(len(works)))
    else:
        try:
            selected_indices = [int(x.strip()) for x in selection.split(",")]
        except ValueError:
            raise ValueError("作品番号は 0,3,5 の形式で入力してください。")

    selected_indices = list(dict.fromkeys(selected_indices))

    for i in selected_indices:
        if i < 0 or i >= len(works):
            raise ValueError(f"作品番号 {i} は範囲外です。")

    print(f"\n{len(selected_indices)}作品をダウンロードします。")

    downloaded = []

    for number, i in enumerate(selected_indices, 1):
        work = works.iloc[i]
        title = str(work["作品名"])
        card_url = str(work["図書カードURL"])

        print(f"[{number}/{len(selected_indices)}] {title}")

        try:
            text = download_work(card_url)
        except Exception as e:
            print(f"  → ERROR: {e}")
            continue

        if text is None:
            print("  → TXTを取得できませんでした。")
            continue

        downloaded.append({
            "title": title,
            "text": text
        })

        print(f"  → OK {len(text):,} chars")

    if not downloaded:
        raise RuntimeError("作品を1件も取得できませんでした。")

    print(f"\nRAWファイルを作成: {RAW_OUTPUT}")

    with open(RAW_OUTPUT, "w", encoding="utf-8") as out:
        for item in downloaded:
            out.write(f"<|title|>{item['title']}\n")
            out.write(f"<|author|>{author_name}\n")
            out.write(item["text"])
            out.write("\n\n<|endoftext|>\n\n")

    original_size = os.path.getsize(RAW_OUTPUT)

    print("\nクリーニングを開始します...")

    total = len(downloaded)
    kept = 0
    total_chars = 0

    with open(CLEAN_OUTPUT, "w", encoding="utf-8") as out:
        for item in tqdm(downloaded):
            cleaned = clean_text(item["text"])

            if cleaned:
                chars = len(cleaned)
                kept += 1
                total_chars += chars

                print(f"Work {kept:2d}: {item['title']} : {chars:,} chars")

                out.write(cleaned)
                out.write("\n\n<|endoftext|>\n\n")
            else:
                print(f"REMOVED: {item['title']}")

    cleaned_size = os.path.getsize(CLEAN_OUTPUT)

    print()
    print("====================================")
    print(f"Author           : {author_name}")
    print(f"Selected works   : {len(selected_indices)}")
    print(f"Downloaded works : {total}")
    print(f"Kept works       : {kept}")
    print(f"Removed works    : {total-kept}")
    print("------------------------------------")
    print(f"Total chars      : {total_chars:,}")
    print(f"Original size    : {original_size/1024:.1f} KB")
    print(f"Cleaned size     : {cleaned_size/1024:.1f} KB")

    if original_size > 0:
        print(f"Reduction        : {100*(1-cleaned_size/original_size):.1f}%")

    print("------------------------------------")
    print(f"RAW output       : {RAW_OUTPUT}")
    print(f"Clean output     : {CLEAN_OUTPUT}")
    print("====================================")

if __name__ == "__main__":
    main()

青空文庫の作品一覧を取得しています...
登録データ数: 19,502
作家名を入力してください: 太宰治

=== 作家候補 ===
  0: 太宰 治 (人物ID=35)
候補が1人なので自動選択します。

選択した作家: 太宰 治 (人物ID=35)

=== 太宰 治 の作品一覧（274作品）===
  0: ア、秋 (作品ID=236)
  1: I can speak (作品ID=1572)
  2: 愛と美について (作品ID=1578)
  3: 青森 (作品ID=46597)
  4: 青森 (作品ID=4357)
  5: 朝 (作品ID=1562)
  6: あさましきもの (作品ID=240)
  7: 新しい形の個人主義 (作品ID=42360)
  8: 兄たち (作品ID=239)
  9: 雨の玉川心中 ― 02 遺書 (作品ID=58054)
 10: 或る忠告 (作品ID=42356)
 11: 老ハイデルベルヒ (作品ID=238)
 12: 『老ハイデルベルヒ』序 (作品ID=54179)
 13: 一日の労苦 (作品ID=1592)
 14: 一問一答 (作品ID=1598)
 15: 一灯 (作品ID=273)
 16: 一歩前進二歩退却 (作品ID=1593)
 17: 田舎者 (作品ID=42362)
 18: 『井伏鱒二選集』後記 (作品ID=42359)
 19: 陰火 (作品ID=272)
 20: ヴィヨンの妻 (作品ID=2253)
 21: 嘘 (作品ID=2254)
 22: 右大臣実朝 (作品ID=2255)
 23: 鬱屈禍 (作品ID=42354)
 24: 姥捨 (作品ID=2256)
 25: 『姥捨』あとがき (作品ID=54169)
 26: 海 (作品ID=42363)
 27: 炎天汗談 (作品ID=43071)
 28: 黄金風景 (作品ID=2257)
 29: 黄村先生言行録 (作品ID=287)
 30: 桜桃 (作品ID=308)
 31: 緒方氏を殺した者 (作品ID=42364)
 32: おさん (作品ID=305)
 33: おしゃれ童子 (作品ID=306)
 34: 織田君の死 (作品ID=42365)
 35: お伽草紙 (作品ID=52380)
 36: お伽草

100%|██████████| 1/1 [00:00<00:00, 928.56it/s]

Work  1: 走れメロス : 9,888 chars

Author           : 太宰 治
Selected works   : 1
Downloaded works : 1
Kept works       : 1
Removed works    : 0
------------------------------------
Total chars      : 9,888
Original size    : 31.5 KB
Cleaned size     : 28.8 KB
Reduction        : 8.3%
------------------------------------
RAW output       : /content/nanoGPT/novel_selected/selected_works.txt
Clean output     : /content/nanoGPT/novel_selected/selected_works_clean.txt


##OPTIONAL 英語文学の作家、作品検索とダウンロード

In [23]:
# ============================================================
# Standard Ebooks 授業用作品ダウンロード
# 日本での利用を考慮して古い英語原著の作家に限定
# GitHub APIの全repository走査は行わない
# ============================================================

import os
import re
import requests
import subprocess
import shutil
from bs4 import BeautifulSoup

BASE = "/content/nanoGPT"
OUT_DIR = os.path.join(BASE, "standardebooks_selected")
WORK_DIR = os.path.join(OUT_DIR, "repos")
RAW_OUTPUT = os.path.join(OUT_DIR, "selected_works.txt")
CLEAN_OUTPUT = os.path.join(OUT_DIR, "selected_works_clean.txt")

os.makedirs(OUT_DIR, exist_ok=True)

# ============================================================
# 授業用作家リスト
# 原則として古い英語原著の作家に限定
# ============================================================

SAFE_AUTHORS = [
    ("William Shakespeare", "william-shakespeare", 1616),
    ("Daniel Defoe", "daniel-defoe", 1731),
    ("Jonathan Swift", "jonathan-swift", 1745),
    ("Samuel Johnson", "samuel-johnson", 1784),
    ("Robert Burns", "robert-burns", 1796),
    ("Jane Austen", "jane-austen", 1817),
    ("William Blake", "william-blake", 1827),
    ("Walter Scott", "walter-scott", 1832),
    ("Edgar Allan Poe", "edgar-allan-poe", 1849),
    ("Emily Brontë", "emily-bronte", 1848),
    ("Mary Shelley", "mary-shelley", 1851),
    ("Charlotte Brontë", "charlotte-bronte", 1855),
    ("Henry David Thoreau", "henry-david-thoreau", 1862),
    ("Elizabeth Gaskell", "elizabeth-gaskell", 1865),
    ("Charles Dickens", "charles-dickens", 1870),
    ("George Eliot", "george-eliot", 1880),
    ("Herman Melville", "herman-melville", 1891),
    ("Walt Whitman", "walt-whitman", 1892),
    ("Robert Louis Stevenson", "robert-louis-stevenson", 1894),
    ("Lewis Carroll", "lewis-carroll", 1898),
    ("Oscar Wilde", "oscar-wilde", 1900),
    ("Mark Twain", "mark-twain", 1910),
    ("O. Henry", "o-henry", 1910),
    ("Bram Stoker", "bram-stoker", 1912),
    ("Henry James", "henry-james", 1916),
    ("Jack London", "jack-london", 1916),
    ("Joseph Conrad", "joseph-conrad", 1924),
    ("Thomas Hardy", "thomas-hardy", 1928),
    ("Arthur Conan Doyle", "arthur-conan-doyle", 1930),
    ("D. H. Lawrence", "d-h-lawrence", 1930),
    ("Rudyard Kipling", "rudyard-kipling", 1936),
    ("H. P. Lovecraft", "h-p-lovecraft", 1937),
    ("Edith Wharton", "edith-wharton", 1937),
    ("F. Scott Fitzgerald", "f-scott-fitzgerald", 1940),
    ("James Joyce", "james-joyce", 1941),
    ("Virginia Woolf", "virginia-woolf", 1941),
]

# ============================================================
# 作家一覧
# ============================================================

print("=" * 70)
print("Standard Ebooks 授業用作家一覧")
print("=" * 70)

for i, (name, slug, death) in enumerate(SAFE_AUTHORS, 1):
    print(f"{i:3d}: {name:<28} 没年 {death}")

print("=" * 70)
print("作家数:", len(SAFE_AUTHORS))
print()

# ============================================================
# 作家選択
# ============================================================

while True:
    try:
        author_no = int(input("作家番号を入力してください: "))
        if 1 <= author_no <= len(SAFE_AUTHORS):
            break
    except ValueError:
        pass
    print("正しい番号を入力してください。")

author_name, author_slug, death_year = SAFE_AUTHORS[author_no - 1]

print()
print("=" * 70)
print("選択した作家:", author_name)
print("没年        :", death_year)
print("=" * 70)

# ============================================================
# GitHub Search API
# 選択した作家だけを検索
# ============================================================

print()
print("Standard Ebooksから作品を検索しています...")

query = f"org:standardebooks {author_slug} in:name"

url = "https://api.github.com/search/repositories"

params = {
    "q": query,
    "per_page": 100
}

headers = {
    "Accept": "application/vnd.github+json",
    "User-Agent": "nanoGPT-Colab"
}

r = requests.get(
    url,
    params=params,
    headers=headers,
    timeout=30
)

if r.status_code == 403:
    raise RuntimeError(
        "GitHub APIのアクセス上限に達しています。\n"
        "しばらく待ってから再実行してください。"
    )

r.raise_for_status()

data = r.json()

items = data.get("items", [])

# ============================================================
# 選択作家のrepositoryだけ残す
# ============================================================

prefix = author_slug + "_"

repos = []

for repo in items:
    name = repo["name"]

    if not name.startswith(prefix):
        continue

    # 翻訳者等が付加されたrepositoryを安全側で除外
    # 基本形式:
    # author_title
    #
    # 翻訳作品では:
    # author_title_translator
    #
    # のようになることがある

    after_author = name[len(prefix):]

    # descriptionからタイトルを取得
    description = repo.get("description") or ""

    m = re.search(
        r"Epub source for the Standard Ebooks edition of (.+?), by ",
        description
    )

    if m:
        title = m.group(1).strip()
    else:
        title = after_author.replace("-", " ").title()

    repos.append({
        "name": name,
        "title": title,
        "url": repo["clone_url"],
        "description": description
    })

# ============================================================
# 重複削除
# ============================================================

unique = {}

for repo in repos:
    unique[repo["name"]] = repo

repos = list(unique.values())

repos.sort(
    key=lambda x: x["title"].lower()
)

if not repos:
    raise RuntimeError(
        f"{author_name} の作品が見つかりませんでした。"
    )

# ============================================================
# 作品一覧
# ============================================================

print()
print("=" * 70)
print(author_name)
print("=" * 70)

for i, repo in enumerate(repos, 1):
    print(f"{i:3d}: {repo['title']}")

print("=" * 70)
print("作品数:", len(repos))
print()

# ============================================================
# 作品選択
# ============================================================

print("作品番号を入力してください。")
print("複数選択例: 1,3,5")
print("全作品    : all")

selection = input("作品番号: ").strip()

if selection.lower() == "all":
    selected = repos
else:
    indices = []
    for x in selection.split(","):
        try:
            n = int(x.strip())
            if 1 <= n <= len(repos):
                indices.append(n - 1)
        except ValueError:
            pass

    selected = [repos[i] for i in indices]

if not selected:
    raise RuntimeError("作品が選択されていません。")

# ============================================================
# 作業フォルダ初期化
# ============================================================

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)

os.makedirs(WORK_DIR, exist_ok=True)

raw_texts = []
clean_texts = []

# ============================================================
# ダウンロード
# ============================================================

for no, work in enumerate(selected, 1):

    print()
    print(
        f"[{no}/{len(selected)}] "
        f"{work['title']}"
    )

    repo_dir = os.path.join(
        WORK_DIR,
        work["name"]
    )

    result = subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            work["url"],
            repo_dir
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    if result.returncode != 0:
        print("ERROR:", result.stderr)
        continue

    # ========================================================
    # Standard Ebooks本文
    # ========================================================

    text_dir = os.path.join(
        repo_dir,
        "src",
        "epub",
        "text"
    )

    if not os.path.isdir(text_dir):
        print("本文フォルダがありません。")
        continue

    xhtml_files = sorted(
        f for f in os.listdir(text_dir)
        if f.endswith(".xhtml")
    )

    # ========================================================
    # 本文ではないファイル
    # ========================================================

    skip_files = {
        "colophon.xhtml",
        "imprint.xhtml",
        "titlepage.xhtml",
        "halftitlepage.xhtml",
        "toc.xhtml",
        "uncopyright.xhtml",
        "dedication.xhtml"
    }

    work_raw = []
    work_clean = []

    # ========================================================
    # XHTMLから本文抽出
    # ========================================================

    for filename in xhtml_files:

        if filename in skip_files:
            continue

        path = os.path.join(
            text_dir,
            filename
        )

        with open(
            path,
            "r",
            encoding="utf-8"
        ) as f:
            html = f.read()

        work_raw.append(html)

        soup = BeautifulSoup(
            html,
            "html.parser"
        )

        # script/style削除
        for tag in soup(
            ["script", "style"]
        ):
            tag.decompose()

        # ====================================================
        # 本文取得
        # ====================================================

        text = soup.get_text(
            separator="\n",
            strip=True
        )

        # ====================================================
        # クリーニング
        # ====================================================

        # 改行コード統一
        text = text.replace(
            "\r\n",
            "\n"
        ).replace(
            "\r",
            "\n"
        )

        # 行内の連続空白
        text = re.sub(
            r"[ \t]+",
            " ",
            text
        )

        # 行頭・行末空白
        text = "\n".join(
            line.strip()
            for line in text.splitlines()
        )

        # 3個以上の改行
        text = re.sub(
            r"\n{3,}",
            "\n\n",
            text
        )

        text = text.strip()

        if text:
            work_clean.append(text)

    # ========================================================
    # 作品単位で結合
    # ========================================================

    raw = "\n\n".join(work_raw)

    clean = "\n\n".join(work_clean)

    raw_texts.append(raw)

    clean_texts.append(clean)

    print(
        "OK:",
        f"{len(clean):,}",
        "characters"
    )

# ============================================================
# 全作品結合
# ============================================================

raw_final = "\n\n".join(raw_texts)

clean_final = "\n\n".join(clean_texts)

# ============================================================
# 保存
# ============================================================

with open(
    RAW_OUTPUT,
    "w",
    encoding="utf-8"
) as f:
    f.write(raw_final)

with open(
    CLEAN_OUTPUT,
    "w",
    encoding="utf-8"
) as f:
    f.write(clean_final)

# ============================================================
# 結果
# ============================================================

print()
print("=" * 70)
print("Author       :", author_name)
print("Death year   :", death_year)
print("Selected     :", len(selected))
print("Characters   :", f"{len(clean_final):,}")
print()
print("RAW output   :", RAW_OUTPUT)
print("Clean output :", CLEAN_OUTPUT)
print("=" * 70)

print()
print("注意:")
print("この作家一覧は授業用に安全側へ限定した候補です。")
print("日本での著作権消滅を法的に保証するものではありません。")
print("翻訳・編集・挿絵等には別個の権利が存在する場合があります。")

Standard Ebooks作品一覧を取得中...
取得済み: 1513
作品repository数: 1513

作家一覧
   1: A (9 works)
   2: Abu al-ʻAlaʼ al-Maʻarri (1 works)
   3: Ada Elizabeth Chesterton (1 works)
   4: Adam Mickiewicz (1 works)
   5: Adam Smith (2 works)
   6: Aeschylus (3 works)
   7: Aesop (1 works)
   8: Agatha Christie (13 works)
   9: Akutagawa Ryūnosuke (1 works)
  10: Alain-René Lesage (1 works)
  11: Alan Sullivan (1 works)
  12: Aldous Huxley (4 works)
  13: Aleksandr Kuprin (3 works)
  14: Alexander Berkman (1 works)
  15: Alexander Hamilton, John Jay, and James Madison (1 works)
  16: Alexander Mackenzie (1 works)
  17: Alexander Pushkin (1 works)
  18: Alexandre Dumas (6 works)
  19: Alfred, Lord Tennyson (1 works)
  20: Algernon Blackwood (1 works)
  21: Algis Budrys (1 works)
  22: Ambrose Bierce (4 works)
  23: Ameen Rihani (2 works)
  24: Anatole France (2 works)
  25: Andre Norton (10 works)
  26: André Gide (1 works)
  27: Angela Brazil (1 works)
  28: Anita Loos (1 works)
  29: Ann Radcliffe (2 work

/tmp/ipykernel_984/760479667.py:199: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(f.read(), "lxml")


  OK: 139,329 characters

完了
Author       : William Shakespeare
Selected     : 1
Downloaded   : 1
Characters   : 139,367
Bytes        : 141,375
Output       : /content/nanoGPT/novel_selected/selected_works_clean.txt


## STEP3 SentencePiece（10,000語）規模のトークナイザを動かしダウンロードした作品に登場するトークンを学習する。

In [71]:
# ============================================================
# 日本語コーパス → SentencePiece → train.bin（最終完成版）あるいは
# 英語コーパス
# ============================================================

import os
import re
import sentencepiece as spm
import numpy as np
import pickle
import threading
import time

BASE = "/content/nanoGPT"
INPUT = "/content/nanoGPT/novel_selected/selected_works_clean.txt"
MODEL_PREFIX = "/content/nanoGPT/jp"
OUT_DIR = os.path.join(BASE, "jp")

# =========================
# 全体タイマー
# =========================
total_start = time.time()

# =========================
# 進捗表示
# =========================
running = True
def heartbeat():
    flag = True
    while running:
        print("ただいま計算中..." if flag else "　　　　　　　　", flush=True)
        flag = not flag
        time.sleep(5)

t = threading.Thread(target=heartbeat)
t.start()

try:
    # =========================
    # ① モデル削除
    # =========================
    t0 = time.time()
    for ext in [".model", ".vocab"]:
        path = MODEL_PREFIX + ext
        if os.path.exists(path):
            os.remove(path)
    print(f"[STEP1] モデル削除時間: {time.time() - t0:.2f} 秒")

    # =========================
    # ② tokenizer学習
    # =========================
    print("=== SentencePiece Training ===")
    t0 = time.time()

    spm.SentencePieceTrainer.train(
        input=INPUT,
        model_prefix=MODEL_PREFIX,
        vocab_size=320000,
        character_coverage=0.9995,
        model_type='bpe',
        num_threads=8,
        input_sentence_size=2000000,   # ←増やした
        shuffle_input_sentence=True,
        max_sentence_length=4096,
        byte_fallback=True,
        hard_vocab_limit=False,
        user_defined_symbols=["<|endoftext|>"]
    )


    print(f"[STEP2] Tokenizer学習時間: {time.time() - t0:.2f} 秒")
    print("Tokenizer DONE")

    # =========================
    # ③ トークナイズ（最重要部分）
    # =========================
    print("=== Encoding ===")
    t0 = time.time()

    sp = spm.SentencePieceProcessor()
    sp.load(MODEL_PREFIX + ".model")

    ids = []
    eot_id = sp.piece_to_id("<|endoftext|>")

    # 全読み込み
    with open(INPUT, encoding="utf-8") as f:
        data = f.read()

    # 元chunk分割
    chunks = data.split("<|endoftext|>")

    # -------------------------
    # 文単位に細分化（超重要）
    # -------------------------
    new_chunks = []

    for chunk in chunks:
        chunk = chunk.strip()
        if not chunk:
            continue

        # 文分割（精度UP）
        sentences = re.split(r"[。！？]", chunk)

        for s in sentences:
            s = s.strip()
            if len(s) > 50:
                new_chunks.append(s + "。")

    # -------------------------
    # encode
    # -------------------------
    for chunk in new_chunks:
        ids.extend(sp.encode(chunk))
        ids.append(eot_id)

    print("Total tokens:", len(ids))
    print(f"[STEP3] Encoding時間: {time.time() - t0:.2f} 秒")

    # =========================
    # ④ train / val 分割
    # =========================
    t0 = time.time()

    n = int(len(ids) * 0.9)
    train_ids = np.array(ids[:n], dtype=np.uint16)
    val_ids   = np.array(ids[n:], dtype=np.uint16)

    print(f"[STEP4] 分割時間: {time.time() - t0:.2f} 秒")

    # =========================
    # ⑤ 保存
    # =========================
    t0 = time.time()

    os.makedirs(OUT_DIR, exist_ok=True)

    train_ids.tofile(os.path.join(OUT_DIR, "train.bin"))
    val_ids.tofile(os.path.join(OUT_DIR, "val.bin"))

    with open(os.path.join(OUT_DIR, "meta.pkl"), "wb") as f:
        pickle.dump({"vocab_size": sp.get_piece_size()}, f)

    print(f"[STEP5] 保存時間: {time.time() - t0:.2f} 秒")

    print("=== ALL DONE ===")
    print("vocab_size =", sp.get_piece_size())

finally:
    running = False
    t.join()

# =========================
# 総時間
# =========================
print(f"\n=== 総処理時間: {time.time() - total_start:.2f} 秒 ===")

ただいま計算中...
[STEP1] モデル削除時間: 0.00 秒
=== SentencePiece Training ===
[STEP2] Tokenizer学習時間: 0.07 秒
Tokenizer DONE
=== Encoding ===
Total tokens: 460
[STEP3] Encoding時間: 0.01 秒
[STEP4] 分割時間: 0.00 秒
[STEP5] 保存時間: 0.00 秒
=== ALL DONE ===
vocab_size = 10942

=== 総処理時間: 5.00 秒 ===


##トークンのベスト100を表示します。

In [72]:
import os
import sentencepiece as spm
from collections import Counter

INPUT = "/content/nanoGPT/novel_selected/selected_works_clean.txt"
TOP_N = 100

models = []
for root, dirs, files in os.walk("/content/nanoGPT"):
    for name in files:
        if name.endswith(".model"):
            models.append(os.path.join(root, name))

if not models:
    raise FileNotFoundError("/content/nanoGPT 以下にSentencePiece .modelがありません。")

print("=== 発見したモデル ===")
for i, path in enumerate(models):
    print(f"{i}: {path}")

if len(models) == 1:
    MODEL = models[0]
    print(f"自動選択: {MODEL}")
else:
    n = int(input("使用するモデル番号: "))
    MODEL = models[n]

sp = spm.SentencePieceProcessor()
sp.load(MODEL)

print(f"\nTokenizer: {MODEL}")
print(f"語彙数: {sp.get_piece_size():,}")

with open(INPUT, "r", encoding="utf-8", errors="ignore") as f:
    text = f.read()

print(f"文字数: {len(text):,}")
print("トークン化しています...")

ids = sp.encode(text, out_type=int)
counts = Counter(ids)
total_tokens = len(ids)

print(f"総トークン数: {total_tokens:,}")
print(f"異なるトークン数: {len(counts):,}")

print(f"\n=== 出現頻度 上位{TOP_N}トークン ===")
print(f"{'順位':>4} {'ID':>7} {'回数':>10} {'割合':>9}  Token")
print("-" * 70)

for rank, (token_id, count) in enumerate(counts.most_common(TOP_N), 1):
    token = sp.id_to_piece(token_id)
    ratio = count / total_tokens * 100
    print(f"{rank:4d} {token_id:7d} {count:10,d} {ratio:8.3f}%  {repr(token)}")

=== 発見したモデル ===
0: /content/nanoGPT/jp.model
自動選択: /content/nanoGPT/jp.model

Tokenizer: /content/nanoGPT/jp.model
語彙数: 10,942
文字数: 9,905
トークン化しています...
総トークン数: 2,856
異なるトークン数: 1,299

=== 出現頻度 上位100トークン ===
  順位      ID         回数        割合  Token
----------------------------------------------------------------------
   1   10184        554   19.398%  '、'
   2   10185        390   13.655%  '。'
   3     264         55    1.926%  '。」'
   4     266         48    1.681%  '▁「'
   5     281         20    0.700%  '私は'
   6     233         18    0.630%  '<0xE5>'
   7     268         15    0.525%  'メロスは'
   8     328         15    0.525%  'ああ'
   9     234         15    0.525%  '<0xE6>'
  10   10186         14    0.490%  'の'
  11     291         13    0.455%  '。「'
  12   10191         11    0.385%  'に'
  13     236         11    0.385%  '<0xE8>'
  14   10205         11    0.385%  'も'
  15     237         10    0.350%  '<0xE9>'
  16     465          9    0.315%  '私を'
  17   10203          9    0.

## STEP4 入力コーパスを学習用と検証用の２つに分けます。

In [73]:
import sentencepiece as spm
import numpy as np
import os
import pickle

print("=== SentencePiece Training ===")


BASE = "/content/nanoGPT"
sp = spm.SentencePieceProcessor()
sp.load(os.path.join(BASE, "jp.model"))


input_file = "/content/nanoGPT/novel_selected/selected_works_clean.txt"
#out_dir = os.path.join(BASE, "jp")
out_dir = os.path.join(BASE, "data", "jp")

os.makedirs(out_dir, exist_ok=True)

print("Reading...")
ids = []
with open(input_file, encoding="utf-8") as f:
    for line in f:
        ids.extend(sp.encode(line))   # ← メモリ安全

print("Encoding done")
print("Total tokens:", len(ids))

n = int(0.9 * len(ids))
train_ids = np.array(ids[:n], dtype=np.uint16)
val_ids = np.array(ids[n:], dtype=np.uint16)

train_ids.tofile(os.path.join(out_dir, "train.bin"))
val_ids.tofile(os.path.join(out_dir, "val.bin"))

with open(os.path.join(out_dir, "meta.pkl"), "wb") as f:
    pickle.dump({"vocab_size": sp.get_piece_size()}, f)
print(f"train tokens = {len(train_ids):,}")
print(f"val tokens   = {len(val_ids):,}")

print("DONE")

=== SentencePiece Training ===
Reading...
Encoding done
Total tokens: 2856
train tokens = 2,570
val tokens   = 286
DONE


## STEP5 GPUが使えるかをチェックします。

In [26]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


## STEP 6 Transformerで学習させます。　CPU環境では --device=cpu とし、 GPUが使えるなら　--device=cuda にする。

In [74]:
## 人間失格規模用

!python train.py \
  --dataset=jp \
  --device=cuda \
  --compile=False \
  --init_from=scratch \
  --n_layer=22 \
  --n_head=16 \
  --n_embd=128 \
  --batch_size=8 \
  --gradient_accumulation_steps=1 \
  --block_size=256 \
  --max_iters=2000 \
  --lr_decay_iters=2000 \
  --warmup_iters=20 \
  --learning_rate=4e-4 \
  --min_lr=3e-5 \
  --eval_interval=20 \
  --eval_iters=20 \
  --dropout=0.1 \
  --log_interval=10 \
  --dtype=float16


Overriding: dataset = jp
Overriding: device = cuda
Overriding: compile = False
Overriding: init_from = scratch
Overriding: n_layer = 22
Overriding: n_head = 16
Overriding: n_embd = 128
Overriding: batch_size = 8
Overriding: gradient_accumulation_steps = 1
Overriding: block_size = 256
Overriding: max_iters = 2000
Overriding: lr_decay_iters = 2000
Overriding: warmup_iters = 20
Overriding: learning_rate = 0.0004
Overriding: min_lr = 3e-05
Overriding: eval_interval = 20
Overriding: eval_iters = 20
Overriding: dropout = 0.1
Overriding: log_interval = 10
Overriding: dtype = float16
tokens per iteration will be: 2,048
found vocab_size = 10942 (inside data/jp/meta.pkl)
Initializing a new model from scratch
number of parameters: 5.73M
/content/nanoGPT/train.py:196: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16'))
num decayed parameter tensors: 90, with

In [76]:
## 走れメロス規模用
!python train.py \
  --dataset=jp \
  --device=cuda \
  --compile=False \
  --init_from=scratch \
  --n_layer=2 \
  --n_head=2 \
  --n_embd=128 \
  --batch_size=4 \
  --gradient_accumulation_steps=1 \
  --block_size=128 \
  --max_iters=3000 \
  --lr_decay_iters=3000 \
  --warmup_iters=20 \
  --learning_rate=3e-4 \
  --min_lr=3e-5 \
  --eval_interval=20 \
  --eval_iters=20 \
  --dropout=0.1 \
  --log_interval=10 \
  --dtype=float16


Overriding: dataset = jp
Overriding: device = cuda
Overriding: compile = False
Overriding: init_from = scratch
Overriding: n_layer = 2
Overriding: n_head = 2
Overriding: n_embd = 128
Overriding: batch_size = 4
Overriding: gradient_accumulation_steps = 1
Overriding: block_size = 128
Overriding: max_iters = 3000
Overriding: lr_decay_iters = 3000
Overriding: warmup_iters = 20
Overriding: learning_rate = 0.0003
Overriding: min_lr = 3e-05
Overriding: eval_interval = 20
Overriding: eval_iters = 20
Overriding: dropout = 0.1
Overriding: log_interval = 10
Overriding: dtype = float16
tokens per iteration will be: 512
found vocab_size = 10942 (inside data/jp/meta.pkl)
Initializing a new model from scratch
number of parameters: 1.79M
/content/nanoGPT/train.py:196: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16'))
num decayed parameter tensors: 10, with 1,8

## STEP7 文章生成をします。

In [77]:
# generate_sp.py
import os
import re
import torch
import sentencepiece as spm
from contextlib import nullcontext

os.chdir("/content/nanoGPT")
from model import GPTConfig, GPT

# =========================
# 設定
# =========================
out_dir = "out"
sp_model_path = "/content/nanoGPT/jp.model"

prompt = "綺麗な衣裳も買って来た。"
num_samples = 1
max_new_tokens = 200

device = "cuda" if torch.cuda.is_available() else "cpu"

repetition_penalty = 1.1
temperature = 0.2
top_k = 20
top_p = 0.9

# =========================
# SentencePiece
# =========================
sp = spm.SentencePieceProcessor()
sp.load(sp_model_path)

def encode(s):
    return sp.encode(s, out_type=int)

def decode(l):
    return sp.decode(l)

# =========================
# 表示整形
# 日本語・英語対応
# =========================
def format_text(text):
    # 日本語
    # 。！？ の後で改行
    # 閉じ括弧 」』）】］” が続く場合も対応
    text = re.sub(
        r'([。！？]+[」』）】］”"]*)[ \t]*',
        r'\1\n',
        text
    )

    # 英語
    # . ! ? の後に空白がある場合に改行
    # " ' などが続く場合にも対応
    text = re.sub(
        r'([.!?]+["\']?)[ \t]+',
        r'\1\n',
        text
    )

    # 行末の余分な空白を削除
    text = re.sub(r'[ \t]+\n', '\n', text)

    # 3個以上の連続改行を2個までにする
    text = re.sub(r'\n{3,}', '\n\n', text)

    return text.strip()

# =========================
# モデル読み込み
# =========================
ckpt_path = os.path.join(out_dir, "ckpt.pt")
checkpoint = torch.load(ckpt_path, map_location=device)

model = GPT(GPTConfig(**checkpoint["model_args"]))
state_dict = checkpoint["model"]

# compile対策
for k in list(state_dict.keys()):
    if k.startswith("_orig_mod."):
        state_dict[k[len("_orig_mod."):]] = state_dict.pop(k)

model.load_state_dict(state_dict)
model.to(device)
model.eval()

ctx = nullcontext()

# =========================
# 生成
# =========================
def generate(model, idx, max_new_tokens):
    block_size = model.config.block_size
    last_tokens = []

    for _ in range(max_new_tokens):
        idx_cond = (
            idx
            if idx.size(1) <= block_size
            else idx[:, -block_size:]
        )

        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature

        # -------------------------
        # repetition penalty
        # -------------------------
        for token in set(idx[0].tolist()):
            logits[0][token] /= repetition_penalty

        # -------------------------
        # top_k
        # -------------------------
        if top_k is not None:
            k = min(top_k, logits.size(-1))
            v, _ = torch.topk(logits, k)
            logits[logits < v[:, [-1]]] = -float("Inf")

        # -------------------------
        # top_p
        # -------------------------
        if top_p is not None:
            sorted_logits, sorted_indices = torch.sort(
                logits,
                descending=True
            )

            cumulative_probs = torch.cumsum(
                torch.softmax(sorted_logits, dim=-1),
                dim=-1
            )

            sorted_indices_to_remove = cumulative_probs > top_p

            sorted_indices_to_remove[..., 1:] = (
                sorted_indices_to_remove[..., :-1].clone()
            )

            sorted_indices_to_remove[..., 0] = False

            indices_to_remove = sorted_indices[
                sorted_indices_to_remove
            ]

            logits[0][indices_to_remove] = -float("Inf")

        probs = torch.softmax(logits, dim=-1)

        # -------------------------
        # 直近tokenの反復防止
        # -------------------------
        for _ in range(10):
            next_token = torch.multinomial(
                probs,
                num_samples=1
            )

            if next_token.item() not in last_tokens:
                break

        # -------------------------
        # 履歴更新
        # -------------------------
        last_tokens.append(next_token.item())

        if len(last_tokens) > 10:
            last_tokens.pop(0)

        # -------------------------
        # token追加
        # -------------------------
        idx = torch.cat(
            (idx, next_token),
            dim=1
        )

    return idx

# =========================
# 実行
# =========================
start_ids = encode(prompt)

x = torch.tensor(
    start_ids,
    dtype=torch.long,
    device=device
)[None, ...]

with torch.no_grad():
    with ctx:
        for _ in range(num_samples):
            y = generate(
                model,
                x,
                max_new_tokens
            )

            text = decode(y[0].tolist())

            # 日本語・英語の文末で改行
            text = format_text(text)

            print("==========")
            print(text)

number of parameters: 1.79M
綺麗な衣裳も買って来た。
さあ、これから行って、村の人たちに知らせて来い。
結婚式は、あすだと。」
メロスは、また、よろよろと歩き出し、家へ帰って神々の祭壇を飾り、祝宴の席を調え、間もなく床に倒れ伏し、呼吸もせぬくらいの深い眠りに落ちてしまった。
眼が覚めたのは夜だった。
メロスは起きてすぐ、花婿の家を訪れた。
そうして、少し事情があるから、結婚式を明日にしてくれ、と頼んだ。
婿の牧人は驚き、こちらには未だ何の仕度も出来ていない、それはいけない、葡萄の季節まで待ってくれ、と答えた。
メロスは、待つことは出来ぬ、どうか明日にしてくれ給え、と更に押してたのんだ。
なかなか承諾してくれない。
夜明けまで議論をつづけて、やっと、どうにか婿をなだめ、すかして、真昼に行われた。
新郎新婦の、神々への宣誓が済んだころ、黒雲が空を覆い、ぽつりぽつり雨が降り出し、やがて車軸を流すような大雨となった。
祝宴に列席していた村人たちは、何か不吉なものを感じたが、それでも、狭い家の中で、めいめい気持を引きたて、むんむん蒸し暑いのも怺え、陽気に歌をうたい、手を拍った。
メロスも、満面に喜色を湛え、しばらくは、王とのあの約束をさえ忘れていた。
祝宴は、夜に入っていよいよ乱れ華やかになり、人々は、外の豪雨を全く気にしなくなった。
メロスは、一生このままここにいたい、と思った。
この佳い人たちと生涯暮して行きたいと願ったが、いまは、自分のからだで、いまは、自分のものでは無い。
ままならぬ事である。
メロスは、わが身に鞭打ち、自分のものでは無い。
あすの日没までには、ついに出発を決意した。
ちょっと一眠りして、まだ十分の時が在る。
あすの日没までには、それからすぐに出発しよう、それからすぐに出発しよう、雨も小降りになっていよう。
その頃には、と考えた。
少しでも永くこの家に愚図愚図とどまっていたかった。
メロスほどの男にも、やはり未練の情というものは在る。
今宵呆然、歓喜に酔っているらしい花嫁に近寄り、 「おめでとう。
私は疲れてしまったから、ちょっとご免こうむって眠りたい。
眼が覚めたら、すぐに市に出かける。
大切な用事があるのだ。
私がいなくても、決して寂しい事は無い。
おまえの兄の、もうおまえには優しい亭主があるのだから、一ばんきらいなもの

In [70]:
import sentencepiece as spm

sp = spm.SentencePieceProcessor()
sp.load("/content/nanoGPT/jp.model")

text = "綺麗な衣裳も買って来た。"

ids = sp.encode(text, out_type=int)
pieces = sp.encode(text, out_type=str)

print("原文 :", text)
print("IDs  :", ids)
print("Pieces:", pieces)
print("unk_id:", sp.unk_id())

print("\n=== 詳細 ===")
for i, (tid, piece) in enumerate(zip(ids, pieces)):
    mark = "  <-- UNK!" if tid == sp.unk_id() else ""
    print(f"{i:2d}: ID={tid:5d} piece={repr(piece)}{mark}")

print("\ndecode:", sp.decode(ids))

原文 : 綺麗な衣裳も買って来た。
IDs  : [29, 1197, 0, 36, 423, 17, 923, 349, 5]
Pieces: ['▁', '綺', '麗', 'な', '衣裳', 'も', '買', 'って来た', '。']
unk_id: 0

=== 詳細 ===
 0: ID=   29 piece='▁'
 1: ID= 1197 piece='綺'
 2: ID=    0 piece='麗'  <-- UNK!
 3: ID=   36 piece='な'
 4: ID=  423 piece='衣裳'
 5: ID=   17 piece='も'
 6: ID=  923 piece='買'
 7: ID=  349 piece='って来た'
 8: ID=    5 piece='。'

decode: 綺 ⁇ な衣裳も買って来た。
